# Delta TV voice — your own voice for every payment

Makes the TV board's announcements in **your voice** with [Chatterbox](https://github.com/resemble-ai/chatterbox) (Resemble AI, MIT licence — free, business use allowed), on Google's free GPU.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.*

1. Run each cell in order (▶ or Shift+Enter).
2. Cell 2 asks for files: **your voice recordings** (WhatsApp `.ogg` voice notes are fine — 10 to 60 seconds of you talking normally, one person, quiet room) and **`voice-lines.json`** (made by `bun src/scripts/voice-clips.ts list`).
3. Cell 3 lets you **listen to a sample** first — change the settings there until it sounds like you.
4. Cell 4 records every line (a few minutes). Cell 5 downloads **`tv_voice_clips.zip`**.
5. On your Mac: unzip it, then `bun src/scripts/voice-clips.ts upload --dir=~/Downloads/tv_voice_clips`.

Only you use this voice; it's yours.

In [ ]:
# 1 — install Chatterbox (2–3 minutes). If Colab asks to "Restart session", do it, then run this cell again.
!nvidia-smi -L || echo "No GPU — set Runtime → Change runtime type → T4 GPU"
!pip -q install chatterbox-tts
!apt-get -qq install -y ffmpeg > /dev/null

In [ ]:
# 2 — upload your voice recordings and voice-lines.json (select them all at once)
import json, os, subprocess
from google.colab import files
up = files.upload()
os.makedirs('samples', exist_ok=True)
voice_files = []
for name, data in up.items():
    if name.endswith('.json'):
        open('voice-lines.json', 'wb').write(data)
    else:
        path = os.path.join('samples', name); open(path, 'wb').write(data); voice_files.append(path)
assert voice_files, 'Upload at least one recording of your voice'
assert os.path.exists('voice-lines.json'), 'Upload voice-lines.json too'
# Your recordings joined into one clean reference clip (mono, 24 kHz, at most 30 seconds, quiet start and end trimmed).
with open('list.txt', 'w') as f:
    for v in voice_files:
        out = v + '.wav'
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', v, '-ac', '1', '-ar', '24000', out], check=True)
        f.write(f"file '{out}'\n")
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'concat', '-safe', '0', '-i', 'list.txt',
                '-af', 'silenceremove=start_periods=1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse',
                '-t', '30', 'reference.wav'], check=True)
lines = json.load(open('voice-lines.json'))['lines']
print(f'{len(voice_files)} recording(s) of you, {len(lines)} lines to record')
from IPython.display import Audio, display
display(Audio('reference.wav'))

In [ ]:
# 3 — load the model and hear a sample. Too flat? raise EXAGGERATION (0.5 → 0.7). Too fast or off? lower CFG (0.5 → 0.3).
EXAGGERATION = 0.55
CFG = 0.45
import torch, torchaudio as ta
from chatterbox.tts import ChatterboxTTS
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ChatterboxTTS.from_pretrained(device=device)
sample = next((l['text'] for l in lines if l['key'].startswith('cs:')), 'Payment collected by Haritha, in Team Money Heist!')
sample += ' D S L P. For two thousand dirhams. Congratulations!'
wav = model.generate(sample, audio_prompt_path='reference.wav', exaggeration=EXAGGERATION, cfg_weight=CFG)
ta.save('sample.wav', wav, model.sr)
print(sample)
display(Audio('sample.wav'))

In [ ]:
# 4 — record every line (picks up where it stopped if run again)
import re, time
os.makedirs('tv_voice_clips', exist_ok=True)
def mp3(src, dst, trim=True):
    af = ['-af', 'silenceremove=start_periods=1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse'] if trim else []
    # The same MP3 settings for every piece, no tags — so the portal can join them one after another.
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src, *af, '-ac', '1', '-ar', '44100', '-codec:a', 'libmp3lame', '-b:a', '128k',
                    '-id3v2_version', '0', '-write_xing', '0', dst], check=True)
manifest = []
# A short pause put between the pieces.
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'lavfi', '-i', 'anullsrc=r=44100:cl=mono', '-t', '0.25',
                '-codec:a', 'libmp3lame', '-b:a', '128k', '-id3v2_version', '0', '-write_xing', '0', 'tv_voice_clips/gap.mp3'], check=True)
manifest.append({'key': 'gap', 'text': '', 'file': 'gap.mp3'})
start = time.time()
for i, l in enumerate(lines, 1):
    name = re.sub(r'[^A-Za-z0-9_-]', '_', l['key']) + '.mp3'
    out = os.path.join('tv_voice_clips', name)
    if not os.path.exists(out):
        wav = model.generate(l['text'], audio_prompt_path='reference.wav', exaggeration=EXAGGERATION, cfg_weight=CFG)
        ta.save('piece.wav', wav, model.sr)
        mp3('piece.wav', out)
    manifest.append({'key': l['key'], 'text': l['text'], 'file': name})
    if i % 20 == 0 or i == len(lines):
        print(f'{i}/{len(lines)} — {int(time.time() - start)} s')
json.dump(manifest, open('tv_voice_clips/manifest.json', 'w'), indent=1)
print('Done. Listen to a few:')
for l in [m for m in manifest if m['key'] != 'gap'][:3]:
    print(l['text']); display(Audio(os.path.join('tv_voice_clips', l['file'])))

In [ ]:
# 5 — download them all
!cd /content && rm -f tv_voice_clips.zip && zip -qr tv_voice_clips.zip tv_voice_clips
files.download('tv_voice_clips.zip')